In [1]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

folder_id = '1d_l-uKZFXudiFpJqBKKkTmblM171meRW'

file_list = drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()

csv_file = None
for file in file_list:
    if file['title'] == 'recenzii_100.csv':
        csv_file = file
        break

if csv_file:
    print(f"Se descarcă: {csv_file['title']}")
    csv_file.GetContentFile('recenzii_100.csv')
else:
    print("Fișierul 'recenzii_100.csv' nu a fost găsit în folder.")

Se descarcă: recenzii_100.csv


In [2]:
import pandas as pd

In [3]:
open_ai_k = 'sk-proj-YvJ9AtlyXweWJLLlfP7nec-4bd83-_azWk2LEI7C4TQLO3EPscMcw9K_4Xm6EU7EWGcmQUDjqoT3BlbkFJBHAY9_HQ9-hcvTgSjCdQBO4RkLj3rfqkqDZodNHxMZxwA59GYKaG6WQchdu6nKlhamK'

In [4]:
open_ai_k += 'LastFewChars'

In [5]:
from openai import OpenAI
import pandas as pd
from tqdm import tqdm


client = OpenAI(api_key=open_ai_key)

df = pd.read_csv("recenzii_100.csv")

few_shot = """
Clasifică următoarea recenzie într-una dintre categoriile: Pozitiv, Negativ, Sugestie.

Exemple:
Recenzie: "Îmi place aplicația, funcționează foarte bine și este ușor de folosit." → Etichetă: Pozitiv
Recenzie: "Se blochează frecvent și interfața e greu de folosit." → Etichetă: Negativ
Recenzie: "Mi-ar plăcea să existe o funcție de export în PDF." → Etichetă: Sugestie
"""


def clasifica(text):
    prompt = f"""{few_shot}

Recenzie: "{text}"
Etichetă:"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Eroare: {e}"

tqdm.pandas()
df["predictie"] = df["text"].progress_apply(clasifica)


100%|██████████| 100/100 [00:46<00:00,  2.17it/s]


In [6]:
df[['label','predictie']].value_counts()

label     predictie         
Sugestie  Sugestie              32
Negativ   Negativ               26
Pozitiv   Pozitiv               19
Negativ   Etichetă: Negativ      8
Pozitiv   Negativ                6
Sugestie  Etichetă: Negativ      2
          Etichetă: Sugestie     2
Negativ   Sugestie               1
Pozitiv   Etichetă: Negativ      1
          Etichetă: Pozitiv      1
          Sugestie               1
Sugestie  Negativ                1
Name: count, dtype: int64